In [1]:
import os
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

# Carregar variáveis de ambiente
load_dotenv()
chat = ChatOpenAI(api_key = os.getenv("OPENAI_API_KEY"),
                  model="gpt-5-mini", temperature=0.7)

 ### O Poder da Composição com LCEL

#### LCEL permite que você aninhe e combine cadeias de forma elegante.
- **`RunnablePassthrough`:** Passa a entrada para a próxima etapa sem modificação. Útil para reencaminhar dados.
- **`RunnablePassthrough.assign()`:** Cria novas chaves no dicionário de entrada, permitindo que a saída de uma mini-cadeia seja usada em prompts futuros.
- **Funções Python como Runnable:** Você pode transformar qualquer função Python em um `Runnable` com `RunnableLambda` (ou simplesmente usando `|` diretamente, o LangChain converte a função automaticamente).

In [4]:
# Cadeia para extrair o sentimento
parser = StrOutputParser()
prompt_sentimento = ChatPromptTemplate.from_template("Analise o seguinte feedback e diga se o sentimento é 'positivo', 'negativo' ou 'neutro': {feedback}")
chain_sentimento = prompt_sentimento | chat | parser

In [5]:
# Cadeia para extrair o nome do produto
prompt_produto = ChatPromptTemplate.from_template("Qual o nome do produto mencionado no seguinte feedback? Responda apenas com o nome do produto: {feedback}")
chain_produto = prompt_produto | chat | parser

### Usaremos o `RunnablePassthrough` para gerenciar as entradas e saídas e combinaremos tudo em uma única `Runnable`.


In [6]:
def gerar_recomendacao(sentimento: str) -> str:
    """Função Python que gera uma recomendação de resposta com base no sentimento."""
    if sentimento.lower() == "positivo":
        return "Responda com gratidão e ofereça um cupom de desconto."
    elif sentimento.lower() == "negativo":
        return "Responda com empatia, peça desculpas e ofereça suporte imediato."
    else:
        return "Agradeça o feedback e peça mais detalhes sobre a experiência."

# A função Python é agora parte da nossa cadeia
runnable_recomendacao = RunnablePassthrough.assign(
    recomendacao=lambda x: gerar_recomendacao(x['sentimento'])
)


### A cadeia completa que orquestra todo o processo
1. Recebe o 'feedback'.
2. Usa RunnablePassthrough.assign para chamar as duas sub-cadeias em paralelo.
3. A saída é um dicionário com 'sentimento' e 'produto'.
4. A saída é passada para a próxima etapa, que é a nossa função Python.
5. A função adiciona a chave 'recomendacao' ao dicionário.

In [7]:
cadeia_completa = (
    RunnablePassthrough.assign(
        sentimento=chain_sentimento,
        produto=chain_produto,
    )
    | runnable_recomendacao
)

In [8]:
feedback_positivo = "Adorei o novo smartphone XPTO. A câmera é incrível e o desempenho é excelente!"
print("--- Análise de Feedback Positivo ---")
resultado_positivo = cadeia_completa.invoke({"feedback": feedback_positivo})

--- Análise de Feedback Positivo ---


In [9]:
print(f"Sentimento: {resultado_positivo['sentimento']}")
print(f"Produto: {resultado_positivo['produto']}")
print(f"Recomendação de Resposta: {resultado_positivo['recomendacao']}")


Sentimento: positivo

Justificativa: termos como "Adorei", "incrível" e "excelente" indicam claramente avaliação favorável.
Produto: XPTO
Recomendação de Resposta: Agradeça o feedback e peça mais detalhes sobre a experiência.


In [10]:
# Exemplo 2: Feedback Negativo
feedback_negativo = "O fone de ouvido 'EarSonic' é péssimo. A bateria acaba em 2 horas e a conexão Bluetooth vive caindo."
print("\n--- Análise de Feedback Negativo ---")
resultado_negativo = cadeia_completa.invoke({"feedback": feedback_negativo})


--- Análise de Feedback Negativo ---


In [11]:
print(f"Sentimento: {resultado_negativo['sentimento']}")
print(f"Produto: {resultado_negativo['produto']}")
print(f"Recomendação de Resposta: {resultado_negativo['recomendacao']}")

Sentimento: negativo

Justificativa: linguagem claramente desfavorável ("péssimo") e queixas específicas sobre bateria curta e conexão instável.
Produto: EarSonic
Recomendação de Resposta: Agradeça o feedback e peça mais detalhes sobre a experiência.
